 # Tabular EDA and Preprocessing Overview

 This notebook performs exploratory data preparation for tabular clinical data. It loads label and VAS sheets from the project data folder,

 standardizes identifiers and visit labeling, reshapes VAS data to a long format, merges labels and scores, creates derived features,

 encodes categorical fields as binary indicators, filters rows with invalid diagnostic or quality flags, and saves a checkpoint of the cleaned dataframe.



 Run the cells in order after the consolidated imports cell at the top. Do not change execution order when reproducing results.

 ### Phase 0: Imports & Notebook Setup



 **Objective:** Provide a consolidated import and environment setup so subsequent cells have consistent dependencies and tools available.



 **Methodology:** Import essential libraries (`os`, `pandas`, `numpy`, `pickle`) once at the top to centralize environment requirements, reduce duplication, and make dependency review straightforward for reproducibility and review.



 **Expected Outcome:** A single, central imports cell that clearly lists runtime dependencies and sets the context for all following data-manipulation steps.

In [1]:
# Consolidated imports required by the notebook
# These libraries are used for filesystem operations, data manipulation, numerical operations, and serialization
import os
import pandas as pd
import numpy as np
import pickle



 ### Phase 1: Data Loading & Header Flattening



 **Objective:** Load the Excel workbook and convert the VAS sheet's two-row header into a single-level header suitable for Pandas operations.



 **Methodology:** Read the labels and VAS sheets with appropriate header arguments, then programmatically collapse multi-level column names (handling "Unnamed" top-level values) into deterministic single strings so column selection and string operations work reliably.



 **Expected Outcome:** Two DataFrames—`df_labels` and `df_vas`—with `df_vas.columns` flattened to single-level names enabling straightforward selection by prefix (e.g., "Baseline_Worst").

In [2]:
# Build the path to the Excel workbook located in the project 'Data' directory
data_dir = os.path.join(os.getcwd(), 'Data')
file_name = 'KP02-Pain maps and VAS 28Jul2022.xlsx'
full_file_path = os.path.join(data_dir, file_name)

# Read label and VAS sheets into dataframes; any I/O error will be printed and re-raised
try:
    df_labels = pd.read_excel(full_file_path, sheet_name='Pain Map Labels', header=0)
    df_vas = pd.read_excel(full_file_path, sheet_name='VAS', header=[0, 1])
except Exception as e:
    print(f"Error loading Excel file: {e}")
    raise

# Convert the VAS sheet's two-level column index into a single-level column index
new_columns = []
for col in df_vas.columns:
    top_level = str(col[0])
    bottom_level = str(col[1])
    if top_level.startswith("Unnamed"):
        new_columns.append(bottom_level)
    else:
        new_columns.append(f"{top_level}_{bottom_level}")

df_vas.columns = new_columns

print("Data loaded and headers flattened!")



Data loaded and headers flattened!


 ### Phase 2: Standardize Patient Identifiers and Column Names



 **Objective:** Normalise patient identifiers (`PN`) and canonicalize column names (including any Hebrew headers) to ensure consistent joins and merges.



 **Methodology:** Detect and rename non-standard header text (e.g., Hebrew "מתנדב" → `PN`), trim and uppercase `PN`, and pad single-digit suffixes with leading zeros using regex so that patient IDs match across image and tabular pipelines.



 **Expected Outcome:** `PN` values in both dataframes are normalized (trimmed, uppercased, and zero-padded where needed), and key columns have consistent names for subsequent merges.

In [3]:
# Locate Hebrew 'PN' header if present and rename it to 'PN' for consistency
for col in df_vas.columns:
    if 'מתנדב' in str(col):
        df_vas.rename(columns={col: 'PN'}, inplace=True)
        break

# Ensure the labels sheet uses 'Visit' as the visit column name
if 'VISIT' in df_labels.columns:
    df_labels.rename(columns={'VISIT': 'Visit'}, inplace=True)

# Normalize PN values in both dataframes: trim, uppercase, and pad single-digit suffixes
df_vas['PN'] = df_vas['PN'].astype(str).str.strip().str.upper()
df_vas['PN'] = df_vas['PN'].str.replace(r'-(\d)$', r'-0\1', regex=True)

df_labels['PN'] = df_labels['PN'].astype(str).str.strip().str.upper()
df_labels['PN'] = df_labels['PN'].str.replace(r'-(\d)$', r'-0\1', regex=True)

print("Patient IDs and column names standardized! 'PN' is ready to use.")



Patient IDs and column names standardized! 'PN' is ready to use.


 ### Phase 3: Reshape VAS Scores to Long Format



 **Objective:** Convert visit-specific, wide-format VAS columns (Baseline, End of treatment, End of trial) into a single long dataframe keyed by `PN` and `Visit`.



 **Methodology:** For each visit prefix, select columns that start with that prefix, strip the prefix to yield common measurement names, attach a `Visit` numeric code, and concatenate the per-visit pieces into a long-form table so each row represents a single patient-visit measurement.



 **Expected Outcome:** `df_vas_long` where each row is a `PN` + `Visit` observation and measurement columns (e.g., `Worst`, `Average`) are standardized across visits for easy merges and modeling.

In [4]:
# Define mapping of visit numeric codes to the VAS column prefixes
visit_mapping = {
    1: 'Baseline',
    2: 'End of treatment',
    3: 'End of trial'
}

# Build a list of long-format pieces, one per visit, then concatenate
vas_long_pieces = []
for visit_num, prefix in visit_mapping.items():
    stage_cols = [col for col in df_vas.columns if col.startswith(prefix)]
    cols_to_keep = ['PN'] + stage_cols

    df_subset = df_vas[cols_to_keep].copy()
    df_subset.columns = ['PN'] + [col.replace(f"{prefix}_", "") for col in stage_cols]

    df_subset['Visit'] = visit_num
    vas_long_pieces.append(df_subset)

df_vas_long = pd.concat(vas_long_pieces, ignore_index=True)

print("VAS data successfully reshaped! 'df_vas_long' is now ready.")



VAS data successfully reshaped! 'df_vas_long' is now ready.


 ### Phase 4: Merge Labels with VAS Scores and Consolidate Laterality



 **Objective:** Left-join label metadata to the long VAS table and unify left/right knee measurements into single `Worst` and `Average` fields.



 **Methodology:** Perform a left merge on `PN` and `Visit` to preserve label rows, interpret `SIDE_` text to decide which side-specific columns (`L-`/`R-`) to use, and copy the appropriate side measurements into unified columns; drop the original side-specific columns to avoid duplication.



 **Expected Outcome:** `df_merged` containing label metadata joined to VAS scores, with `Worst` and `Average` reflecting the correct anatomical side for each row.

In [5]:
# Merge label information with the long-format VAS scores; preserve all rows from df_labels
df_merged = pd.merge(df_labels, df_vas_long, on=['PN', 'Visit'], how='left')

# Create boolean masks for side interpretation based on the textual SIDE_ column
is_right = df_merged['SIDE_'].astype(str).str.strip().str.title() == 'Right'
is_left = df_merged['SIDE_'].astype(str).str.strip().str.title() == 'Left'

# For rows labeled 'Right', copy right-specific measurements into unified columns
df_merged.loc[is_right, 'Worst'] = df_merged.loc[is_right, 'R-Worst']
df_merged.loc[is_right, 'Average'] = df_merged.loc[is_right, 'R-Average']

# For rows labeled 'Left', copy left-specific measurements into unified columns
df_merged.loc[is_left, 'Worst'] = df_merged.loc[is_left, 'L-Worst']
df_merged.loc[is_left, 'Average'] = df_merged.loc[is_left, 'L-Average']

# Remove the original side-specific columns when present
cols_to_drop = ['R-Worst', 'L-Worst', 'R-Average', 'L-Average']
df_merged.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Left merge and side-consolidation complete!")

print(f"Post-Merge: Dataset contains {len(df_merged)} total records across {df_merged['PN'].nunique()} unique patients.")

Left merge and side-consolidation complete!
Post-Merge: Dataset contains 492 total records across 82 unique patients.


 ### Phase 5: Encode Laterality as a Binary Feature



 **Objective:** Convert the textual `SIDE_` indicator to a numeric binary column `side_right` for modeling.



 **Methodology:** Map `SIDE_` values (`Right` → 1, `Left` → 0), fill missing values conservatively, cast to integer, and drop the original textual field to prevent ambiguity between string and numeric representations.



 **Expected Outcome:** `df_merged` contains `side_right` (1 for right, 0 for left) alongside the consolidated `Worst` and `Average` measurements; `SIDE_` is removed.

In [6]:
# If the SIDE_ column exists, map textual values to binary and drop the original column
if 'SIDE_' in df_merged.columns:
    df_merged['side_right'] = df_merged['SIDE_'].astype(str).str.strip().str.title().map({
        'Right': 1,
        'Left': 0
    })

    df_merged['side_right'] = df_merged['side_right'].fillna(0).astype(int)
    df_merged.drop(columns=['SIDE_'], inplace=True)
    print("Success: 'SIDE_' text removed. Created 'side_right' binary column (1=Right, 0=Left).")
else:
    print("Check: 'SIDE_' column not found. It may have already been processed.")

# Show a small sample to verify the transformation
display(df_merged[['PN', 'side_right', 'Worst', 'Average']].head(10))



Success: 'SIDE_' text removed. Created 'side_right' binary column (1=Right, 0=Left).


,PN,side_right,Worst,Average
0,MC-02,1,70.0,53
1,MC-02,0,46.0,53.0
2,AB-03,1,0.0,0
3,AB-03,0,66.0,39.0
4,YE-04,1,0.0,0
5,YE-04,0,99.0,66.0
6,AP-05,1,82.0,58
7,AP-05,0,0.0,0.0
8,CH-07,1,27.0,13
9,CH-07,0,62.0,56.0


 ### Phase 6: Encode Sex as a Binary Indicator



 **Objective:** Transform the `SEX` code into a consistent binary `male` column compatible with downstream models.



 **Methodology:** Map the original coded values to a binary representation (preserving the intended numerical semantics), fill missing entries with a safe default, cast to integer, and drop the original `SEX` column to keep the schema compact.



 **Expected Outcome:** `df_merged` includes `male` (binary integer) and no longer contains `SEX`, enabling direct use in modeling and analysis.

In [7]:
# Convert the 'SEX' code into a binary 'male' column and remove the original column
if 'SEX' in df_merged.columns:
    df_merged['male'] = df_merged['SEX'].map({1: 0, 2: 1})
    df_merged['male'] = df_merged['male'].fillna(0).astype(int)
    df_merged.drop(columns=['SEX'], inplace=True)
    print("Success: Converted 'SEX' (1/2) into 'male' (1/0).")
    display(df_merged[['PN', 'male']].head(10))
else:
    print("Check: 'SEX' column not found. It may have already been renamed or dropped.")



Success: Converted 'SEX' (1/2) into 'male' (1/0).


,PN,male
0,MC-02,1
1,MC-02,1
2,AB-03,1
3,AB-03,1
4,YE-04,1
5,YE-04,1
6,AP-05,1
7,AP-05,1
8,CH-07,1
9,CH-07,1


 ### Phase 7: Rename Anterior Columns for Consistency



 **Objective:** Standardize anterior-region column names to the short, consistent naming convention used by the image pipeline.



 **Methodology:** Apply a deterministic rename mapping (e.g., `Anterior_A5` → `Ant_5`) to make column names compact and predictable for later feature engineering and cross-referencing with image-derived features.



 **Expected Outcome:** `df_merged` with anterior-region columns renamed to the `Ant_*` convention, aligning tabular naming with image-processing conventions.

In [8]:
# Apply a deterministic rename mapping for anterior columns to concise names
df_merged = df_merged.rename(columns={
    "Anterior_3": "Ant_3",
    "Anterior_A5": "Ant_5",
    "Anterior_A6": "Ant_6",
    "Anterior_A7": "Ant_7",
    "Anterior_A9": "Ant_9"
})
print("Anterior columns renamed successfully.")



Anterior columns renamed successfully.


 ### Phase 8: Unify Lateral and Medial Indicators



 **Objective:** Merge split indicator variants representing lateral and medial anatomy into single boolean/integer flags.



 **Methodology:** Use bitwise OR (or equivalent aggregation) across the split indicator columns (e.g., `Lateral_LJL` | `Lateral_LCL`) after filling missing values to ensure any positive signal is preserved; then drop the original split columns to reduce redundancy.



 **Expected Outcome:** New unified `Lateral` and `Medial` columns (0/1 integers) in `df_merged`, with redundant component columns removed.

In [9]:
# Combine lateral indicator variants using bitwise OR to preserve any positive signal
if 'Lateral_LJL' in df_merged.columns and 'Lateral_LCL' in df_merged.columns:
    df_merged['Lateral'] = (
        df_merged['Lateral_LJL'].fillna(0).astype(int) |
        df_merged['Lateral_LCL'].fillna(0).astype(int)
    )

# Combine medial indicator variants similarly
if 'Medial_MJL' in df_merged.columns and 'Medial_MCL' in df_merged.columns:
    df_merged['Medial'] = (
        df_merged['Medial_MJL'].fillna(0).astype(int) |
        df_merged['Medial_MCL'].fillna(0).astype(int)
    )

# Remove the now-redundant split columns if present
df_merged.drop(
    columns=['Lateral_LJL', 'Lateral_LCL', 'Medial_MJL', 'Medial_MCL'],
    inplace=True,
    errors='ignore'
)

print("Created unified 'Lateral' and 'Medial' columns and removed the original split columns.")
display(df_merged.head())



Created unified 'Lateral' and 'Medial' columns and removed the original split columns.


,PN,PC,INCLUDED,GROUP,DOCTOR,QUALITY,PF_DIAGNOSIS,Visit,AKP,Ant_3,...,Ant_6,Ant_7,Ant_9,תאריך,Worst,Average,side_right,male,Lateral,Medial
0,MC-02,2,0,1.0,Spizer,1.0,1,1,1,1,...,0,0,1,27.03.17,70.0,53,1,1,1,0
1,MC-02,2,0,1.0,Spizer,1.0,1,1,1,1,...,0,1,0,27.03.17,46.0,53.0,0,1,0,0
2,AB-03,3,0,2.0,Barzilay,1.0,1,1,0,0,...,0,0,0,3.05.17,0.0,0,1,1,0,0
3,AB-03,3,0,2.0,Barzilay,1.0,1,1,1,0,...,1,0,0,3.05.17,66.0,39.0,0,1,0,1
4,YE-04,4,0,2.0,Barzilay,1.0,1,1,0,0,...,0,0,0,8.05.17,0.0,0,1,1,0,0


 ### Phase 9: Create Target Label `pain_label`



 **Objective:** Derive the modeling target `pain_label` from the existing `AKP` column in a numeric format.



 **Methodology:** Validate presence of `AKP`, coerce it to numeric (handling non-numeric gracefully by converting to NaN), and assign the result to `pain_label` so target distribution and class balance can be assessed.



 **Expected Outcome:** `df_merged` with a numeric `pain_label` column ready for model training and value-count inspection.

In [10]:
# Ensure the chosen target column 'AKP' is present and create a numeric 'pain_label'
if 'AKP' not in df_merged.columns:
    raise KeyError("df_merged must contain 'AKP' to create pain_label.")

df_merged['pain_label'] = pd.to_numeric(df_merged['AKP'], errors='coerce')

print("Created 'pain_label' from 'AKP'.")
print("Value counts (including NaN):")
print(df_merged['pain_label'].value_counts(dropna=False).sort_index())

display(df_merged[['PN', 'Visit', 'AKP', 'pain_label']].head(10))

# Check how many valid labels we have
valid_labels = df_merged['pain_label'].notna().sum()
print(f"Post-Labeling: {valid_labels} valid pain labels found. Distribution: {df_merged['pain_label'].value_counts().to_dict()}")

Created 'pain_label' from 'AKP'.
Value counts (including NaN):
pain_label
0    127
1    365
Name: count, dtype: int64


,PN,Visit,AKP,pain_label
0,MC-02,1,1,1
1,MC-02,1,1,1
2,AB-03,1,0,0
3,AB-03,1,1,1
4,YE-04,1,0,0
5,YE-04,1,1,1
6,AP-05,1,1,1
7,AP-05,1,0,0
8,CH-07,1,0,0
9,CH-07,1,1,1


Post-Labeling: 492 valid pain labels found. Distribution: {1: 365, 0: 127}


 ### Phase 10: Filter Rows with Invalid Diagnosis or Quality Flags



 **Objective:** Remove rows where `PF_DIAGNOSIS` or `QUALITY` indicate invalid or low-quality records.



 **Methodology:** Coerce both filter columns to numeric, build a boolean removal mask where either value equals zero, and subset the dataframe to retain only high-quality diagnostic rows; report counts removed for transparency.



 **Expected Outcome:** A filtered `df_merged` containing only rows with valid diagnosis and quality flags, reducing noise and ensuring analytic integrity.

In [11]:
# Validate that df_merged exists and contains required filter columns
if 'df_merged' not in globals() or not isinstance(df_merged, pd.DataFrame):
    raise NameError("df_merged is not available. Run the preprocessing cells first.")

required_cols = ['PF_DIAGNOSIS', 'QUALITY']
missing_cols = [c for c in required_cols if c not in df_merged.columns]
if missing_cols:
    raise KeyError(f"df_merged is missing required columns: {missing_cols}")

before_rows = len(df_merged)

pf_diag_num = pd.to_numeric(df_merged['PF_DIAGNOSIS'], errors='coerce')
quality_num = pd.to_numeric(df_merged['QUALITY'], errors='coerce')

remove_mask = pf_diag_num.eq(0) | quality_num.eq(0)
df_merged = df_merged.loc[~remove_mask].reset_index(drop=True)

removed_rows = int(remove_mask.sum())
after_rows = len(df_merged)

print('=== Filtering Summary ===')
print(f"Removed rows (PF_DIAGNOSIS == 0 OR QUALITY == 0): {removed_rows}")
print(f"Rows before: {before_rows}")
print(f"Rows after: {after_rows}")



=== Filtering Summary ===
Removed rows (PF_DIAGNOSIS == 0 OR QUALITY == 0): 60
Rows before: 492
Rows after: 432


 ### Phase 11: Create Derived Anterior Pattern Features



 **Objective:** Add higher-level anatomical pattern features (`Pat`, `Ext`) derived from combinations of anterior indicator columns.



 **Methodology:** Ensure `Ant_*` columns are numeric and non-null, then compute `Pat` (positive when at least two of `Ant_5`, `Ant_6`, `Ant_7` are present) and `Ext` (positive when at least two of `Ant_3`, `Ant_6`, `Ant_9` are present) to capture clinically relevant anterior symptom patterns.



 **Expected Outcome:** `df_merged` augmented with integer `Pat` and `Ext` features that summarize spatial anterior patterns for modeling and subgroup analysis.

In [12]:
# Ensure specific anterior columns are numeric and default missing values to 0
ant_cols = ["Ant_3", "Ant_5", "Ant_6", "Ant_7", "Ant_9"]
for col in ant_cols:
    if col in df_merged.columns:
        df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce").fillna(0).astype(int)

# Derive 'Pat' when at least two of Ant_5, Ant_6, Ant_7 are positive
if all(col in df_merged.columns for col in ["Ant_5", "Ant_6", "Ant_7"]):
    df_merged["Pat"] = (
        df_merged[["Ant_5", "Ant_6", "Ant_7"]].sum(axis=1) >= 2
    ).astype(int)

# Derive 'Ext' when at least two of Ant_3, Ant_6, Ant_9 are positive
if all(col in df_merged.columns for col in ["Ant_3", "Ant_6", "Ant_9"]):
    df_merged["Ext"] = (
        df_merged[["Ant_3", "Ant_6", "Ant_9"]].sum(axis=1) >= 2
    ).astype(int)

print("Pat and Ext columns were added to df_merged.")
display(df_merged[["PN", "Visit", "side_right", "Ant_3", "Ant_5", "Ant_6", "Ant_7", "Ant_9", "Pat", "Ext"]].head())



Pat and Ext columns were added to df_merged.


,PN,Visit,side_right,Ant_3,Ant_5,Ant_6,Ant_7,Ant_9,Pat,Ext
0,MC-02,1,1,1,1,0,0,1,0,1
1,MC-02,1,0,1,1,0,1,0,1,0
2,AB-03,1,1,0,0,0,0,0,0,0
3,AB-03,1,0,0,0,1,0,0,0,0
4,YE-04,1,1,0,0,0,0,0,0,0


 ### Phase 12: Reorder Columns for Readability and Run Visit Completeness Diagnostics



 **Objective:** Reorder columns so related anatomical and derived features appear together, and compute per-`PN`+`Visit` row counts to identify missing knee entries.



 **Methodology:** Move prioritized columns (`Lateral`, `Medial`, `Pat`, `Ext`) immediately after `Ant_9` for coherence; then group by `PN` and `Visit` to count rows per visit and surface combinations that have only one row (indicating a missing contralateral entry).



 **Expected Outcome:** A human-friendly column order in `df_merged` and a short diagnostic summary showing visits with both knees (2 rows) versus visits with one knee (1 row), enabling quality checks before modeling.

In [13]:
# Reorder dataframe columns so that prioritised features follow Ant_9
priority_cols = ["Lateral", "Medial", "Pat", "Ext"]

if "Ant_9" in df_merged.columns:
    cols = list(df_merged.columns)
    cols = [col for col in cols if col not in priority_cols]
    ant9_index = cols.index("Ant_9")
    new_cols = cols[:ant9_index + 1] + priority_cols + cols[ant9_index + 1:]
    new_cols = [col for col in new_cols if col in df_merged.columns]
    df_merged = df_merged[new_cols]

print("Columns reordered successfully.")
display(df_merged.head())


Columns reordered successfully.


,PN,PC,INCLUDED,GROUP,DOCTOR,QUALITY,PF_DIAGNOSIS,Visit,AKP,Ant_3,...,Lateral,Medial,Pat,Ext,תאריך,Worst,Average,side_right,male,pain_label
0,MC-02,2,0,1.0,Spizer,1.0,1,1,1,1,...,1,0,0,1,27.03.17,70.0,53,1,1,1
1,MC-02,2,0,1.0,Spizer,1.0,1,1,1,1,...,0,0,1,0,27.03.17,46.0,53.0,0,1,1
2,AB-03,3,0,2.0,Barzilay,1.0,1,1,0,0,...,0,0,0,0,3.05.17,0.0,0,1,1,0
3,AB-03,3,0,2.0,Barzilay,1.0,1,1,1,0,...,0,1,0,0,3.05.17,66.0,39.0,0,1,1
4,YE-04,4,0,2.0,Barzilay,1.0,1,1,0,0,...,0,0,0,0,8.05.17,0.0,0,1,1,0


In [16]:
# Calculate how many rows exist for each PN + Visit combination
pn_visit_counts = df_merged.groupby(['PN', 'Visit']).size().reset_index(name='row_count')

# Count how many combinations have exactly 1 row, 2 rows, etc.
distribution = pn_visit_counts['row_count'].value_counts().sort_index()

print("\n=== Row Count Distribution per Visit ===")
for count, num_visits in distribution.items():
    if count == 2:
        print(f"✅ Visits with exactly 2 rows (Both knees present): {num_visits}")
    elif count == 1:
        print(f"⚠️ Visits with exactly 1 row (One knee missing): {num_visits}")
    else:
        print(f"❓ Visits with {count} rows: {num_visits}")

# Isolate the visits that only have 1 row to see which ones they are
single_row_visits = pn_visit_counts[pn_visit_counts['row_count'] == 1]

if not single_row_visits.empty:
    print(f"\nSample of visits missing a knee (Only 1 row):")
    display(single_row_visits.head(10))
# Final dataset snapshot
print(f"Final Stats: {len(df_merged)} records | {df_merged['PN'].nunique()} patients | Visits: {sorted(df_merged['Visit'].unique().tolist())}")



=== Row Count Distribution per Visit ===
⚠️ Visits with exactly 1 row (One knee missing): 2
✅ Visits with exactly 2 rows (Both knees present): 215

Sample of visits missing a knee (Only 1 row):


,PN,Visit,row_count
178,SZ-68,2,1
179,SZ-68,3,1


Final Stats: 432 records | 76 patients | Visits: [1, 2, 3]


 ### Phase 13: Save Cleaned Checkpoint



 **Objective:** Persist the cleaned and processed dataframe to disk for fast reuse and reproducibility.



 **Methodology:** Serialize `df_merged` to a binary checkpoint (pickle) under a clear filename so subsequent analysis or modeling steps can load a preprocessed snapshot without re-running the full pipeline.



 **Expected Outcome:** A file `cleaned_tabular_checkpoint.pkl` containing the finalized `df_merged`, allowing deterministic loading of the preprocessed tabular dataset for downstream tasks.



 Note: The following cell executes the binary export. Do not modify its code to preserve reproducibility.

In [15]:
# Export the cleaned dataframe to a pickle file for later reuse
with open('cleaned_tabular_checkpoint.pkl', 'wb') as f:
    pickle.dump(df_merged, f)
print("DataFrame exported to cleaned_tabular_checkpoint.pkl")

DataFrame exported to cleaned_tabular_checkpoint.pkl
